# Performance by variant complexity

The four rows report phenomenal-model recovery (P), unconditional mechanism-probe recovery (M), mechanism recovery conditional on P being correct (M|P), and mechanism recovery conditional on P being incorrect ($M\mid\neg P$). Missing or failed experiments count as failures for P and for all of their expected mechanism probes, so each run uses a fixed denominator of 80 (Core) or 512 (Full). Repeated runs are summarized by computing each run's rate first and then averaging the rates.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import yaml
from yumeow_plot import get_fig

REPO = Path.cwd()
if not (REPO / 'tasks' / 'task_split.yaml').is_file():
    REPO = REPO.parent
RESULTS = REPO / 'results'
OUTPUT_DIR = REPO / 'logs' / 'analysis' / '260925分析结果'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DOMAINS = ('physics', 'chemistry', 'biology', 'material')
PANEL_KEYS = (*DOMAINS, 'overall')
PANEL_TITLES = ('Physics', 'Chemistry', 'Biology', 'Materials', 'Overall')
COMPLEXITIES = np.arange(6)

split = yaml.safe_load((REPO / 'tasks' / 'task_split.yaml').read_text())
CORE_IDENTITIES = {
    (group['domain'], task_name)
    for group in split['core_set']
    for task_name in group['task_names']
}

CORE_CONFIGS = [
    dict(agent='PySR', base='no LLM', runs=[('pysr',)]),
    dict(agent='Direct-Ask based on PySR', base='Deepseek-v4-flash-0731',
         runs=[('pysr_direct_ask_deepseek-v4-flash-0731',)]),
    dict(agent='Direct-Ask based on P', base='Deepseek-v4-flash-0731',
         runs=[('reference_phenomenology_direct_ask_deepseek-v4-flash-0731',)]),
    dict(agent='Codex', base='GPT-5.6-sol',
         runs=[('codex-gpt-5_6-xhigh-coreset-r1',)]),
    dict(agent='Codex', base='GLM-5.3-flash', runs=[
        ('codex-cjl-glm-5.3-flash-high-fullset-r1',),
        ('codex-cjl-glm-5.3-flash-high-coreset-r2',),
        ('codex-cjl-glm-5.3-flash-high-coreset-r3',),
    ]),
    dict(agent='ClaudeCode', base='GLM-5.3-flash',
         runs=[('claudecode-glm5_3flash-high-coreset',)]),
    dict(agent='Codex', base='Deepseek-v4-flash-0731', runs=[
        ('codex-cjl-deepseek-v4-flash-0731-low-coreset-r1',),
        ('codex-cjl-deepseek-v4-flash-0731-low-coreset-r2',),
        ('codex-cjl-deepseek-v4-flash-0731-low-coreset-r3',),
    ]),
    dict(agent='ClaudeCode', base='Deepseek-v4-flash-0731',
         runs=[('claudecode-deepseekv4flash0731-low-coreset',)]),
    dict(agent='DSH', base='Deepseek-v4-flash-0731',
         runs=[('dsh_deepseek-v4-flash-0731',)]),
    dict(agent='DSH', base='Deepseek-v4-pro-0813',
         runs=[('dsh_deepseek-v4-pro-0813',)]),
]

FULL_CONFIGS = [
    dict(agent='PySR', base='no LLM', runs=[('pysr',)]),
    dict(agent='Direct-Ask based on PySR', base='Deepseek-v4-flash-0731',
         runs=[('pysr_direct_ask_deepseek-v4-flash-0731',)]),
    dict(agent='Direct-Ask based on P', base='Deepseek-v4-flash-0731',
         runs=[('reference_phenomenology_direct_ask_deepseek-v4-flash-0731',)]),
    dict(agent='Codex', base='GPT-5.6-sol', runs=[(
        'codex-gpt-5_6-xhigh-coreset-r1',
        'codex-gpt-5_6-xhigh-fullset',
    )]),
    dict(agent='Codex', base='GLM-5.3-flash',
         runs=[('codex-cjl-glm-5.3-flash-high-fullset-r1',)]),
]


In [ ]:
def variant_complexity(task_name: str) -> int:
    suffix = task_name.rsplit(' - ', 1)[-1]
    if suffix == 'Original':
        return 0
    if not suffix.startswith('Variant '):
        raise ValueError(f'Cannot parse variant complexity: {task_name}')
    return len(suffix.removeprefix('Variant ').split('-'))


TASK_METADATA = {}
FULL_GROUPS = {(group['domain'], group['task_group']) for group in split['full_set']}
for task_path in (REPO / 'tasks').glob('*/*/*.yaml'):
    task = yaml.safe_load(task_path.read_text())
    domain, task_group = task_path.relative_to(REPO / 'tasks').parts[:2]
    if (domain, task_group) not in FULL_GROUPS:
        continue
    TASK_METADATA[(domain, task['task_name'])] = dict(
        probes=len(task['mechanism_probes']),
        complexity=variant_complexity(task['task_name']),
    )
assert len(TASK_METADATA) == 512


def load_run(experiments: tuple[str, ...], selected=None):
    records = []
    seen = set()
    for experiment in experiments:
        root = RESULTS / experiment
        for path in root.glob('*/*/*.json'):
            domain = path.relative_to(root).parts[0]
            payload = json.loads(path.read_text())
            evaluation = payload.get('evaluation')
            task_name = (evaluation.get('task_name')
                         if isinstance(evaluation, dict) else path.stem)
            identity = (domain, task_name)
            if selected is not None and identity not in selected:
                continue
            if identity in seen:
                raise ValueError(f'Duplicate task in one run: {identity}')
            seen.add(identity)
            if not isinstance(evaluation, dict):
                continue
            phenomenal = bool(evaluation.get('phenomenal', {})
                              .get('ood_test', {})
                              .get('numerically_equivalent'))
            probes = evaluation.get('mechanism_probes', [])
            probe_passes = sum(bool(probe.get('scores', {})
                                      .get('ood_test', {})
                                      .get('numerically_equivalent'))
                               for probe in probes)
            records.append(dict(
                domain=domain, task_name=task_name,
                complexity=variant_complexity(task_name),
                phenomenal=phenomenal, probes=len(probes),
                probe_passes=probe_passes,
            ))
    expected_identities = set(TASK_METADATA) if selected is None else selected
    recorded_identities = {(r['domain'], r['task_name']) for r in records}
    for domain, task_name in expected_identities - recorded_identities:
        metadata = TASK_METADATA[(domain, task_name)]
        records.append(dict(
            domain=domain, task_name=task_name,
            complexity=metadata['complexity'],
            phenomenal=False, probes=metadata['probes'], probe_passes=0,
        ))
    if len(records) != len(expected_identities):
        raise ValueError(f'Unexpected run coverage: {len(records)}/{len(expected_identities)}')
    return records


def one_run_rates(records, panel_key, complexity):
    subset = [r for r in records
              if r['complexity'] == complexity
              and (panel_key == 'overall' or r['domain'] == panel_key)]
    p_rate = (sum(r['phenomenal'] for r in subset) / len(subset)
              if subset else np.nan)
    all_probe_count = sum(r['probes'] for r in subset)
    m_rate = (sum(r['probe_passes'] for r in subset) / all_probe_count
              if all_probe_count else np.nan)
    p_correct = [r for r in subset if r['phenomenal']]
    p_probe_count = sum(r['probes'] for r in p_correct)
    mp_rate = (sum(r['probe_passes'] for r in p_correct) / p_probe_count
               if p_probe_count else np.nan)
    p_incorrect = [r for r in subset if not r['phenomenal']]
    not_p_probe_count = sum(r['probes'] for r in p_incorrect)
    m_not_p_rate = (sum(r['probe_passes'] for r in p_incorrect) / not_p_probe_count
                    if not_p_probe_count else np.nan)
    return p_rate, m_rate, mp_rate, m_not_p_rate


def finite_mean(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    return float(values.mean()) if len(values) else 0.0


def summarize_configs(configs, selected, expected):
    series = []
    for config in configs:
        runs = [load_run(run, selected=selected) for run in config['runs']]
        label = f"{config['agent']} ({config['base']})"
        metrics = {'P': {}, 'M': {}, 'M|P': {}, 'M|~P': {}}
        for panel_key in PANEL_KEYS:
            p_values, m_values, mp_values, m_not_p_values = [], [], [], []
            for complexity in COMPLEXITIES:
                run_rates = [one_run_rates(run, panel_key, complexity) for run in runs]
                p_values.append(finite_mean([value[0] for value in run_rates]))
                m_values.append(finite_mean([value[1] for value in run_rates]))
                mp_values.append(finite_mean([value[2] for value in run_rates]))
                m_not_p_values.append(finite_mean([value[3] for value in run_rates]))
            metrics['P'][panel_key] = np.asarray(p_values)
            metrics['M'][panel_key] = np.asarray(m_values)
            metrics['M|P'][panel_key] = np.asarray(mp_values)
            metrics['M|~P'][panel_key] = np.asarray(m_not_p_values)
        series.append(dict(agent=config['agent'], label=label, metrics=metrics))
    return series


core_series = summarize_configs(CORE_CONFIGS, CORE_IDENTITIES, 80)
full_series = summarize_configs(FULL_CONFIGS, None, 512)
[(item['label']) for item in core_series], [(item['label']) for item in full_series]


In [ ]:
COLORS = ('#d33f6a', '#4a6fe3', '#bb65b6', '#71a451', '#5f99ce', '#c57b3e')
MARKERS = ('o', 's', '^', 'D', 'v', 'P', 'X', '>', '<', '*')
DOTTED_AGENTS = {'PySR', 'Direct-Ask based on PySR', 'Direct-Ask based on P'}


def plot_complexity_panels(series, output_stem):
    fig, axes = get_fig(RN=4, CN=5, FW=28, A_ratio=1, HS=0.5, VS=0.5, LM=5, RM=5, TM=5, BM=5, dpi=300, fontsize=14, lw=1, sharex=True, sharey=True)
    ax_loader = iter(axes)
    legend_by_label = {}
    for metric in ('P', 'M', 'M|P', 'M|~P'):
        for panel_index, (panel_key, panel_title) in enumerate(zip(PANEL_KEYS, PANEL_TITLES)):
            ax = next(ax_loader)
            for index, item in enumerate(series):
                if metric == 'P' and item['agent'] in {
                    'Direct-Ask based on P', 'Direct-Ask based on PySR'
                }:
                    continue
                if metric != 'P' and item['agent'] == 'PySR':
                    continue
                line, = ax.plot(
                    COMPLEXITIES, item['metrics'][metric][panel_key],
                    color=COLORS[index % len(COLORS)],
                    marker=MARKERS[index % len(MARKERS)],
                    linestyle=':' if item['agent'] in DOTTED_AGENTS else '-',
                    linewidth=1.4, markersize=4, label=item['label'],
                )
                legend_by_label.setdefault(item['label'], line)
            if metric == 'P':
                ax.set_title(panel_title, fontsize=14)
            if metric == 'M|~P':
                ax.set_xlabel('Variant complexity', fontsize=10)
            if panel_index == 0:
                metric_label = r'$M\mid\neg P$' if metric == 'M|~P' else metric
                ax.set_ylabel(f'{metric_label} (%)', fontsize=14)
            ax.set_xlim(-0.15, 5.15)
            ax.set_ylim(-0.03, 1.03)
            ax.set_xticks(COMPLEXITIES)
            ax.set_yticks(np.linspace(0, 1, 6))
            ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
            ax.tick_params(axis='both', labelsize=10)
            ax.grid(True, color='#d9d9d9', linewidth=0.7, alpha=0.8)
    legend_labels = [item['label'] for item in series if item['label'] in legend_by_label]
    fig.legend(
        [legend_by_label[label] for label in legend_labels], legend_labels,
        loc='lower center', bbox_to_anchor=(0.5, 0.0),
        ncol=2, frameon=False, fontsize=10,
        handlelength=2.4, columnspacing=1.5,
    )
    fig.subplots_adjust(bottom=0.18, top=0.96)
    for extension in ('png', 'svg', 'pdf'):
        fig.savefig(
            f'./logs/analysis/260925分析结果/{output_stem}.{extension}',
            dpi=300, bbox_inches='tight',
        )
    return fig


In [ ]:
core_fig = plot_complexity_panels(
    core_series, 'core_variant_complexity_P_M_given_P'
)


In [ ]:
full_fig = plot_complexity_panels(
    full_series, 'full_variant_complexity_P_M_given_P'
)
